# NB03 — Entraînement LSTM / Transformer sur Colab GPU (Semaine 3)

**Objectif** : Lancer les 6 runs (LSTM × 3 seeds + Transformer × 3 seeds) sur les splits verrouillés v02, avec persistance sur Google Drive.

**Prérequis** :
- Compte Google + Google Drive activé
- Repo `Projet_Peptide` sur GitHub (public ou token PAT si privé)
- Branche `main` à jour : contient `data/splits/*_v02.csv` + `configs/*_base.yaml` (schéma imbriqué)

**Durée estimée Colab GPU T4** :
- LSTM : ~15-20 min / run × 3 = 45-60 min
- Transformer : ~30-45 min / run × 3 = 90-135 min
- **Total ≈ 2-3 h** → une seule session

In [ ]:
# 1. Monter Google Drive (persistance)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Dossier de travail persistant sur Drive
import os, sys, subprocess, json, time, shutil
WORK = '/content/drive/MyDrive/Projet_Peptide'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(f'Travail dans : {WORK}')
print(f'Contenu : {os.listdir(WORK)}')

In [ ]:
# 2. Cloner / mettre à jour le repo
REPO_URL = 'https://github.com/FEDI-HASSINE/Projet_Peptide.git'
REPO_DIR = os.path.join(WORK, 'Projet_Peptide')

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Clone du repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo existant → fetch + checkout main')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], check=True)

os.chdir(REPO_DIR)
print(f'Dans {REPO_DIR}')
print(f'Branche : {subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()}')
print(f'Commit : {subprocess.run(["git", "rev-parse", "--short=7", "HEAD"], capture_output=True, text=True, cwd=REPO_DIR).stdout.strip()}')

In [ ]:
# 3. Vérifier les artefacts verrouillés (splits v02 + configs)
required = [
    'data/splits/train_v02.csv',
    'data/splits/val_v02.csv',
    'data/splits/test_v02.csv',
    'configs/lstm_base.yaml',
    'configs/transformer_base.yaml',
    'requirements.txt',
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f'Artefacts manquants sur main : {missing}')
print('Tous les artefacts verrouillés présents')

# Afficher tailles splits
import pandas as pd
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/splits/{split}_v02.csv')
    print(f'   {split}_v02.csv : {len(df):,} séquences')

In [ ]:
# 4. Installer les dépendances + vérifier GPU
print('Installation requirements...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

# Vérifier torch + CUDA
import torch
print(f'Python : {sys.version.split()[0]}')
print(f'Torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cpu':
    print('ATTENTION : pas de GPU détecté ! Runtime -> Changer le type d'exécution -> GPU T4')

In [ ]:
# 5. Configuration des runs
RUNS = [
    {'model': 'lstm', 'seed': 42, 'config': 'configs/lstm_base.yaml'},
    {'model': 'lstm', 'seed': 43, 'config': 'configs/lstm_base.yaml'},
    {'model': 'lstm', 'seed': 44, 'config': 'configs/lstm_base.yaml'},
    {'model': 'transformer', 'seed': 42, 'config': 'configs/transformer_base.yaml'},
    {'model': 'transformer', 'seed': 43, 'config': 'configs/transformer_base.yaml'},
    {'model': 'transformer', 'seed': 44, 'config': 'configs/transformer_base.yaml'},
]

# Dossier du repo (où train.py écrit runs.csv et checkpoints)
CKPT_BASE = os.path.join(REPO_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(REPO_DIR, 'results', 'week3')
os.makedirs(CKPT_BASE, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
RUNS_CSV = os.path.join(RESULTS_DIR, 'runs.csv')

print(f'Checkpoints -> {CKPT_BASE}')
print(f'Runs CSV    -> {RUNS_CSV}')
print(f'{len(RUNS)} runs planifiés')

In [ ]:
# 6. Fonction d'entraînement unique avec resume + logging
def run_one(model_name, seed, config_path, resume=True):
    """Lance un entraînement, gère resume, écrit runs.csv incrémental."""
    ckpt_dir = os.path.join(CKPT_BASE, model_name, f'seed_{seed}')
    best_ckpt = os.path.join(ckpt_dir, 'best.pt')

    cmd = [
        sys.executable, '-m', 'peplib.train',
        '--config', config_path,
        '--seed', str(seed),
        # --out-dir retiré : le config YAML définit train.checkpoints_dir
    ]
    if resume and os.path.exists(best_ckpt):
        cmd.append('--resume')
        print(f'Resume depuis {best_ckpt}')

    print(f'{model_name.upper()} seed={seed}')
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
    elapsed = time.time() - t0

    # Afficher sortie (dernières lignes)
    for line in result.stdout.strip().split('\n')[-5:]:
        print(f'   {line}')
    if result.stderr:
        for line in result.stderr.strip().split('\n')[-3:]:
            print(f'   WARNING {line}')

    # Lire le runs.csv mis à jour par peplib.train (append)
    if os.path.exists(RUNS_CSV):
        df = pd.read_csv(RUNS_CSV)
        last = df[(df['model']==model_name) & (df['seed']==seed)].tail(1)
        if not last.empty:
            print(f'   val_pp={last["val_pp"].values[0]:.2f} | epoch={int(last["epoch"].values[0])} | {elapsed/60:.1f} min')

    return result.returncode == 0

In [ ]:
# 7. Boucle principale — lancer les runs séquentiellement
from tqdm.auto import tqdm

print('='*60)
print('DEBUT DES 6 RUNS')
print('='*60)

success = []
failed = []

for run in tqdm(RUNS, desc='Runs globaux'):
    ok = run_one(run['model'], run['seed'], run['config'])
    if ok:
        success.append(f"{run['model']}-{run['seed']}")
    else:
        failed.append(f"{run['model']}-{run['seed']}")
    # Pause pour laisser respirer le GPU
    time.sleep(5)

print('\n' + '='*60)
print('RESUME')
print('='*60)
print(f'Reussis ({len(success)}) : {success}')
print(f'Echoues ({len(failed)}) : {failed}')

In [ ]:
# 8. Afficher le tableau final runs.csv + agrégation
RUNS_CSV = os.path.join(RESULTS_DIR, 'runs.csv')

if os.path.exists(RUNS_CSV):
    df = pd.read_csv(RUNS_CSV)
    best = df.loc[df.groupby(['model', 'seed'])['val_pp'].idxmin()]
    print('\nMeilleur val_pp par run :')
    print(best[['model', 'seed', 'epoch', 'val_pp', 'val_loss', 'elapsed_s']].to_string(index=False))

    agg = best.groupby('model')['val_pp'].agg(['mean', 'std', 'count']).round(4)
    print('\nAgrégation par modèle (seeds 42,43,44) :')
    print(agg.to_string())

    # Sauvegarder une copie horodatée sur Drive
    ts = time.strftime('%Y%m%d-%H%M%S')
    backup = os.path.join(WORK, 'results', 'week3', f'runs_{ts}.csv')
    os.makedirs(os.path.dirname(backup), exist_ok=True)
    shutil.copy2(RUNS_CSV, backup)
    print(f'\nBackup runs.csv -> {backup}')
else:
    print('Aucun runs.csv trouvé (échec avant écriture)')

In [ ]:
# 9. Commandes utiles pour reprise manuelle
print('''
POUR REPRENDRE UN RUN INTERROMPU :
   # Dans une nouvelle session Colab :
   # 1. Monter Drive + cd WORK (cellules 1-2)
   # 2. Installer deps (cellule 4)
   # 3. Lancer UN SEUL run avec --resume (auto si best.pt existe) :
   !python -m peplib.train --config configs/lstm_base.yaml --seed 43 \
       --out-dir /content/drive/MyDrive/Projet_Peptide/checkpoints/lstm/seed_43 --resume

POUR LANCER SEULEMENT LES TRANSFORMERS (si LSTM déjà faits) :
   # Modifier la liste RUNS dans la cellule 5 pour ne garder que transformer 42/43/44
   # Puis ré-exécuter cellules 6-8

POUR RÉDUIRE LE TEMPS (test rapide) :
   # Dans configs/*.yaml : max_epochs: 10  (au lieu de 30)
   # Ou passer --max-epochs si peplib.train le supporte
''')

---

## Notes importantes pour Colab GPU T4

1. **Timeout** : session max ~12 h (souvent 8-9 h réel). Les 6 runs complets ≈ 2-3 h → une seule session suffit.
2. **Reprise** : le script `peplib.train` supporte `--resume` depuis `best.pt`. Les checkpoints sont sur Drive → ils survivent à la déconnexion.
3. **Mémoire** : Colab T4 ≈ 16 Go VRAM. Batch size 128 (config) est OK. Si OOM → réduire `batch_size` dans les configs.
4. **Progression** : `runs.csv` est écrit à chaque epoch (append) → visible en temps réel dans l'onglet Drive.
5. **Seeds** : 42, 43, 44 → agrégation moyenne ± std attendue pour le benchmark mois 8 (budget égal).
6. **Splits** : `data/splits/*_v02.csv` sont les artefacts verrouillés A6 (90/5/5 par familles, F(0,90)=0). Ne pas les modifier.

**Gate de validation Semaine 3** : `val_pp < 19.49` (baseline unigramme, PR #20) sur chaque run.